# import Data

In [105]:
import numpy as np
import kagglehub
import pandas as pd
import os

import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler

from keras.layers import Dense, Flatten, LSTM, Dropout
from keras.models import Sequential
from keras.layers import SpatialDropout1D
from tensorflow.keras import activations

from sklearn.metrics import mean_squared_error

In [106]:
path = kagglehub.dataset_download("nitirajkulkarni/bangkok-th-1609350")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'bangkok-th-1609350' dataset.
Path to dataset files: /kaggle/input/bangkok-th-1609350


In [107]:
file_path = os.path.join(path, "air_quality_historical.csv")
df = pd.read_csv(file_path)
df.head()

,date,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index,us_aqi,european_aqi
0,2022-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-08-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-08-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-08-04,18.758824,12.876471,328.058824,17.723529,8.682353,44.235294,0.125882,0.0,1.697059,NaN,NaN
4,2022-08-05,16.229167,11.100000,256.250000,13.625000,6.650000,37.208333,0.135833,0.0,0.916667,46.117647,23.117647


# Prepare training data

In [108]:
df['date'] = pd.to_datetime(df['date'])
df_2025 = df[(df['date'].dt.year == 2025) & (df['date'].dt.month > 8)]
df = df_2025
df.head()

,date,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index,us_aqi,european_aqi
1127,2025-09-01,9.379167,8.845833,1295.125000,16.395833,8.195833,48.458333,0.147917,0.000000,2.218750,32.875000,22.916667
1128,2025-09-02,11.229167,10.612500,1565.583333,18.912500,10.170833,46.666667,0.163750,0.333333,1.252083,42.333333,24.250000
1129,2025-09-03,14.016667,11.958333,969.083333,20.279167,9.804167,48.500000,0.204583,2.041667,2.389583,41.583333,26.916667
1130,2025-09-04,18.016667,14.687500,1177.750000,22.454167,10.729167,48.083333,0.267917,4.416667,1.620833,55.916667,31.333333
1131,2025-09-05,17.700000,15.191667,947.583333,21.129167,12.387500,50.875000,0.290417,3.708333,2.122917,56.333333,34.041667


In [109]:
fig_pm2_5 = go.Figure()
fig_pm2_5.add_trace(go.Scatter(x=df['date'], y=df['pm2_5'], mode='lines', name='PM2.5 Actual'))
fig_pm2_5.update_layout(title='PM2.5 Values Over Time (2025)', xaxis_title='Date', yaxis_title='PM2.5 Value')
fig_pm2_5.show()

In [110]:
X=np.array(df['pm2_5'])

In [111]:
scaler = MinMaxScaler(feature_range=(0, 1))
X_normalized = scaler.fit_transform(np.array([X]).T)

In [112]:
X_train=X_normalized[:-1]
Y_train=X_normalized[1:]

In [113]:
lag = 3

In [114]:
ind=np.arange(lag)
ind=np.arange(X.shape[0]-lag).reshape(X.shape[0]-lag,1)+ind
X_train=X_normalized[ind]
X_train=X_train.reshape((X_train.shape[0],1,X_train.shape[1]))

In [115]:
X_train.shape

(119, 1, 3)

In [116]:
X_train

array([[[0.10357174, 0.14059907, 0.16880622]],

       [[0.14059907, 0.16880622, 0.22600646]],

       [[0.16880622, 0.22600646, 0.23657323]],

       [[0.22600646, 0.23657323, 0.28320671]],

       [[0.23657323, 0.28320671, 0.17622915]],

       [[0.28320671, 0.17622915, 0.03274823]],

       [[0.17622915, 0.03274823, 0.05466771]],

       [[0.03274823, 0.05466771, 0.04008384]],

       [[0.05466771, 0.04008384, 0.09562484]],

       [[0.04008384, 0.09562484, 0.47960877]],

       [[0.09562484, 0.47960877, 0.51305563]],

       [[0.47960877, 0.51305563, 0.30530085]],

       [[0.51305563, 0.30530085, 0.15727884]],

       [[0.30530085, 0.15727884, 0.18260414]],

       [[0.15727884, 0.18260414, 0.13981312]],

       [[0.18260414, 0.13981312, 0.12147411]],

       [[0.13981312, 0.12147411, 0.03947254]],

       [[0.12147411, 0.03947254, 0.        ]],

       [[0.03947254, 0.        , 0.11544843]],

       [[0.        , 0.11544843, 0.26076325]],

       [[0.11544843, 0.26076325, 0.05737

In [117]:
Y_train=X_train[:,:,0][1:,:]
X_train=X_train[:-1,:,:]

# Train Model using LSTM

In [118]:
model = Sequential()
model.add(LSTM(16, input_shape=(1, lag)))
model.add(Dense(4))
model.add(Dense(1))
model.compile(loss='mean_squared_error', optimizer='adam')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



In [119]:
epochs = 100
batch_size = 5

history = model.fit(X_train, Y_train, epochs=epochs, batch_size=batch_size,validation_split=0.1)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 0.1575 - val_loss: 0.1973
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0625 - val_loss: 0.0770
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0288 - val_loss: 0.0356
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0225 - val_loss: 0.0301
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0205 - val_loss: 0.0246
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0184 - val_loss: 0.0246
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0161 - val_loss: 0.0205
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0139 - val_loss: 0.0179
Epoch 9/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0121 - val_loss: 0.0150
Epoch 10/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0102 - val_loss: 0.0136
Epoch 11/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0086 - val_loss: 0.0103
Epoch 12/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.

In [120]:
Y_show=model.predict(X_train)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step


# Result

In [121]:
Y_train_actual = scaler.inverse_transform(Y_train)
Y_show_actual = scaler.inverse_transform(Y_show)

rmse = np.sqrt(mean_squared_error(Y_train_actual, Y_show_actual))
print(f"Root Mean Squared Error (RMSE): {rmse}")

Root Mean Squared Error (RMSE): 0.2146730233105747


In [122]:
fig=go.Figure()
fig.add_trace(go.Scatter(x=[i for i in range(len(Y_train.T[0]))],y=scaler.inverse_transform(Y_train).T[0],name='Actual'))
fig.add_trace(go.Scatter(x=[i for i in range(len(Y_train.T[0]))],y=scaler.inverse_transform(Y_show).T[0],name='Predicted'))
fig.show()